In [1]:
from pathlib import Path
import pandas as pd

In [2]:
data_dir = Path("../data/raw")

In [3]:
files = sorted(data_dir.glob("*.csv"))

print(f"Total CSV files:  {len(files)}")
for file in files:
    print(file.name)

Total CSV files:  9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [4]:
data_files = []

for file in files:
    df = pd.read_csv(file)

    data_files.append({
        "file" : file.name,
        "rows" : df.shape[0],
        "columns" : df.shape[1],
        "columns_names" : list(df.columns)
    })

df_data_files = pd.DataFrame(data_files)
df_data_files


,file,rows,columns,columns_names
0,olist_customers_dataset.csv,99441,5,"[customer_id, customer_unique_id, customer_zip..."
1,olist_geolocation_dataset.csv,1000163,5,"[geolocation_zip_code_prefix, geolocation_lat,..."
2,olist_order_items_dataset.csv,112650,7,"[order_id, order_item_id, product_id, seller_i..."
3,olist_order_payments_dataset.csv,103886,5,"[order_id, payment_sequential, payment_type, p..."
4,olist_order_reviews_dataset.csv,99224,7,"[review_id, order_id, review_score, review_com..."
5,olist_orders_dataset.csv,99441,8,"[order_id, customer_id, order_status, order_pu..."
6,olist_products_dataset.csv,32951,9,"[product_id, product_category_name, product_na..."
7,olist_sellers_dataset.csv,3095,4,"[seller_id, seller_zip_code_prefix, seller_cit..."
8,product_category_name_translation.csv,71,2,"[product_category_name, product_category_name_..."


In [5]:
for file in files:
    print(file.name)
    df = pd.read_csv(file)
    display(df.head(3))
    print("-"*80)

olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


--------------------------------------------------------------------------------
olist_geolocation_dataset.csv


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP


--------------------------------------------------------------------------------
olist_order_items_dataset.csv


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


... (9 outputs left out)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


--------------------------------------------------------------------------------
product_category_name_translation.csv


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


--------------------------------------------------------------------------------


In [6]:
orders = pd.read_csv(data_dir / "olist_orders_dataset.csv")
items = pd.read_csv(data_dir / "olist_order_items_dataset.csv")
payments = pd.read_csv(data_dir / "olist_order_payments_dataset.csv")

for name, df in [
    ("orders", orders),
    ("items", items),
    ("payments", payments),
]:
    print(
        name,
        "| rows:", len(df),
        "| unique order_id:", df["order_id"].nunique(),
        "| repeated order_id:", df["order_id"].duplicated().sum(),
    )

orders | rows: 99441 | unique order_id: 99441 | repeated order_id: 0
items | rows: 112650 | unique order_id: 98666 | repeated order_id: 13984
payments | rows: 103886 | unique order_id: 99440 | repeated order_id: 4446


In [7]:
order_ids = set(orders["order_id"])
item_ids = set(items["order_id"])
payment_ids = set(payments["order_id"])

print("Orders without items:", len(order_ids - item_ids))
print("Orders without payments:", len(order_ids - payment_ids))
print("Items without an order:", len(item_ids - order_ids))
print("Payments without an order:", len(payment_ids - order_ids))

Orders without items: 775
Orders without payments: 1
Items without an order: 0
Payments without an order: 0


In [9]:
orders_order_ids = set(orders["order_id"])
items_order_ids = set(items["order_id"])
payments_order_ids = set(payments["order_id"])

print("Orders without items:",
      len(orders_order_ids - items_order_ids))

print("Orders without payments:",
      len(orders_order_ids - payments_order_ids))

print("Item rows referencing unknown orders:",
      len(items_order_ids - orders_order_ids))

print("Payment rows referencing unknown orders:",
      len(payments_order_ids - orders_order_ids))

Orders without items: 775
Orders without payments: 1
Item rows referencing unknown orders: 0
Payment rows referencing unknown orders: 0


In [10]:
orders.loc[
    orders["order_id"].isin(orders_order_ids - items_order_ids),
    "order_status"
].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [11]:
orders.loc[
    orders["order_id"].isin(orders_order_ids - payments_order_ids),
    ["order_id", "order_status", "order_purchase_timestamp"]
]

,order_id,order_status,order_purchase_timestamp
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38


In [12]:
order_id = "bfbd0f9bdef84302105ad712db648a6c"

items.loc[
    items["order_id"] == order_id,
    ["order_id", "order_item_id", "price", "freight_value"]
]

,order_id,order_item_id,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,44.99,2.83


In [ ]:
item_totals = (
    items.assign(item_total=items["price"] + items["freight_value"])
    .groupby("order_id", as_index=False)["item_total"]
    .sum()
)

payment_totals = (
    payments.groupby("order_id", as_index=False)["payment_value"]
    .sum()
)


                               order_id  payment_value
0      00010242fe8c5a6d1ba2dd792cb16214          72.19
1      00018f77f2f0320c557190d7a144bdd3         259.83
2      000229ec398224ef6ca0657da4fc703e         216.87
3      00024acbcdf0a6daa1e931b038114c75          25.78
4      00042b26cf59d7ce69dfabb4e55b4fd9         218.04
...                                 ...            ...
99435  fffc94f6ce00a00581880bf54a75a037         343.40
99436  fffcd46ef2263f404302a634eb57f7eb         386.53
99437  fffce4705a9662cd70adb13d4a31832d         116.85
99438  fffe18544ffabc95dfada21779c9644f          64.71
99439  fffe41c64501cc87c801fd61db3f6244          55.79

[99440 rows x 2 columns]


In [33]:
reconciliation = orders[["order_id", "order_status"]].merge(
    item_totals,
    on="order_id",
    how="left",
).merge(
    payment_totals,
    on="order_id",
    how="left",
)

missing_payments = reconciliation[
    reconciliation["item_total"].notna()
    & reconciliation["payment_value"].isna()
]

missing_payments

,order_id,order_status,item_total,payment_value
30710,bfbd0f9bdef84302105ad712db648a6c,delivered,143.46,NaN


In [34]:
with_both_totals = reconciliation.dropna(
    subset=["item_total", "payment_value"]
).copy()

with_both_totals["difference"] = (
    with_both_totals["payment_value"] - with_both_totals["item_total"]
).round(2)

mismatches = with_both_totals[
    with_both_totals["difference"].abs() > 0.01
]

print("Orders with both totals:", len(with_both_totals))
print("Orders with different totals:", len(mismatches))

mismatches[
    ["order_id", "order_status", "item_total", "payment_value", "difference"]
].head(10)

Orders with both totals: 98665
Orders with different totals: 303


,order_id,order_status,item_total,payment_value,difference
464,8adafb3466daa5395694d3a906ff9d40,delivered,218.02,218.00,-0.02
867,bb2e64c3040ceb9b7ca2bfc602adca08,delivered,257.36,257.34,-0.02
1080,84d6d9710c8af32b5e88f2d1c14ab871,delivered,57.68,61.70,4.02
1126,74016effecaa79d592487f6a4ee47d4b,delivered,51.51,56.96,5.45
1669,239f380355f65dcb68551f07d16fc4a8,delivered,222.63,251.63,29.00
1729,4c57f545143e8865ca2347d8cba154a7,delivered,139.61,151.01,11.40
1986,6e57e23ecac1ae881286657694444267,delivered,350.41,333.91,-16.50
2277,b38b3526b8b8fdc807e8a0a42ab78573,delivered,30.06,30.19,0.13
2357,051fcda88d997d3ff86012da2a556342,delivered,56.60,51.70,-4.90
2541,8b5058499c412c6cf8d013de40e4f9d2,delivered,48.77,51.02,2.25


In [35]:
sample_ids = [
    "8adafb3466daa5395694d3a906ff9d40",
    "239f380355f65dcb68551f07d16fc4a8",
]

display(items.loc[
    items["order_id"].isin(sample_ids),
    ["order_id", "order_item_id", "price", "freight_value"]
])

display(payments.loc[
    payments["order_id"].isin(sample_ids),
    ["order_id", "payment_sequential", "payment_type", "payment_value"]
])

,order_id,order_item_id,price,freight_value
15591,239f380355f65dcb68551f07d16fc4a8,1,149.0,73.63
60921,8adafb3466daa5395694d3a906ff9d40,1,21.0,6.25
60922,8adafb3466daa5395694d3a906ff9d40,2,35.0,10.42
60923,8adafb3466daa5395694d3a906ff9d40,3,21.0,6.25
60924,8adafb3466daa5395694d3a906ff9d40,4,35.0,10.42
60925,8adafb3466daa5395694d3a906ff9d40,5,28.0,8.34
60926,8adafb3466daa5395694d3a906ff9d40,6,28.0,8.34


,order_id,payment_sequential,payment_type,payment_value
2030,239f380355f65dcb68551f07d16fc4a8,1,credit_card,251.63
32737,8adafb3466daa5395694d3a906ff9d40,1,credit_card,218.00
